**Note: Requires Databricks Serverless GPU A10 Compute with AI v4 Environment**

# Dependency Installation

In [0]:
%pip install --upgrade pip
dbutils.library.restartPython()

%pip install "textacy==0.13.0" fastcoref huggingface_hub ta
dbutils.library.restartPython()

import spacy
spacy.prefer_gpu()
# Download English model
import spacy.cli; spacy.cli.download("en_core_web_sm")
dbutils.library.restartPython()

# Spark Session

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.config("spark.sql.session.timeZone", "UTC").appName("FinSentAnalysis").getOrCreate()

# Data Curation

## Financial News Data

In [0]:
WORKSPACE = "paid"
VOLUME = f'/Volumes/{WORKSPACE}/default/ensf612/'

In [0]:
news_df = spark.read.json(f"dbfs:{VOLUME}aapl_news.json").select(*['id', 'created', 'title', 'teaser', 'body'])
news_df.limit(10).display()

### TimeStamp Conversion

In [0]:
news_df = news_df.withColumn('created', regexp_replace('created', r"^[A-Za-z]{3},\s+", "")).withColumn('created', to_timestamp('created', "dd MMM yyyy HH:mm:ss Z")).orderBy('created').withColumnRenamed('created', 'date')
news_df.limit(10).display()

### Text Preprocessing

1. Remove HTML Tags
2. Remove New Line, Tab, Carriage Return
3. Replace URL, Emails, Phone Numbers, Emojis, Hashtags, Social User Handles
4. Normalize Bullet Points, Quotation Marks, Multi Line Hyphenation, and White Spaces
5. Remove 'Image' and 'Also Read:..'

In [0]:
# Remove HTML Tags
from bs4 import BeautifulSoup as bs

def parse_html(text: str) -> str:
  return bs(text, 'html.parser').get_text()

# Remove New Line, Tab, Carriage Return
import re
def remove_carriage(text: str) -> str:
  return re.sub(r'\r|\n|\t', ' ', text)

# Remove 'Image' and 'Also Read'
def replace_irrelevant(text: str) -> str:
    return re.sub(r'Image:.*|Also Read: ', '', text)

# Create a Textacy pipeline
import networkx
from textacy.preprocessing import make_pipeline
from textacy.preprocessing.replace import emails, emojis, hashtags, phone_numbers, urls, user_handles
from textacy.preprocessing.normalize import bullet_points, quotation_marks, hyphenated_words, whitespace
text_pipe = make_pipeline(
    parse_html,
    remove_carriage,
    emails,
    emojis,
    hashtags,
    phone_numbers,
    urls,
    user_handles,
    bullet_points,
    quotation_marks,
    hyphenated_words,
    whitespace,
    replace_irrelevant
    )

# Convert into a Spark UDF
def text_preprocessing(text: str) -> str:
  return text_pipe(text)

In [0]:
pd_df = news_df.toPandas()
pd_df['title'] = pd_df['title'].apply(text_preprocessing)
pd_df['teaser'] = pd_df['teaser'].apply(text_preprocessing)
pd_df['body'] = pd_df['body'].apply(text_preprocessing)
news_df = spark.createDataFrame(pd_df)
news_df.limit(10).display()

### Coreference Resolution

#### Local Model Cache

In [0]:
from huggingface_hub import snapshot_download
import os

local_tmp = "/tmp/hf_models/fcoref"
os.makedirs(local_tmp, exist_ok=True)

model_tmp = snapshot_download(
    "biu-nlp/f-coref",
    local_dir=local_tmp
)

#### Coreference Function

In [0]:
from fastcoref import FCoref
import pandas as pd
import numpy as np

HF_COREF_CACHE_DIR = str(model_tmp)

_coref = None

def get_coref():
    """
    Lazily initialize FCoref once per worker process.
    Called inside the pandas UDF.
    """
    global _coref
    if _coref is None:
        _coref = FCoref(
            model_name_or_path=HF_COREF_CACHE_DIR,
            device="cuda:0",
        )
    return _coref

def get_resolved_text(result) -> str:

    if result is None:
        return None

    """
    Build a "resolved" text by replacing later mentions in each cluster
    with the first mention's surface string.
    """
    text = result.text
    clusters = result.get_clusters(as_strings=False)  # [[(start, end), ...], ...]

    # Collect replacements: (start, end, replacement_text)
    replacements = []

    for cluster in clusters:
        if not cluster:
            continue

        # First span is the canonical mention
        canonical_start, canonical_end = cluster[0]
        canonical_text = text[canonical_start:canonical_end]

        # Replace all *later* mentions with canonical text
        for (start, end) in cluster[1:]:
            replacements.append((start, end, canonical_text))

    # Sort by start index so we can rebuild left→right
    replacements.sort(key=lambda x: x[0])

    # Rebuild the text with replacements applied
    resolved_parts = []
    cur = 0

    for start, end, rep in replacements:
        # add text before this mention
        resolved_parts.append(text[cur:start])
        # add canonical form
        resolved_parts.append(rep)
        # move cursor
        cur = end

    # add the tail of the text
    resolved_parts.append(text[cur:])

    return "".join(resolved_parts)


def coreference_resolution(text: str) -> str:

    if text is None or text == "":
        return None

    return get_resolved_text(get_coref().predict(text))

#### Implementation

In [0]:
%%capture
pd_df = news_df.toPandas()
pd_df["body"] = pd_df["body"].apply(coreference_resolution)
pd_df["title"] = pd_df["title"].apply(coreference_resolution)
pd_df["teaser"] = pd_df["teaser"].apply(coreference_resolution)
news_df = spark.createDataFrame(pd_df)
news_df.limit(10).display()

### Contextual Sentence Segmentation

#### Segmentation Function

In [0]:
import en_core_web_sm

nlp = en_core_web_sm.load()

APPLE_NAMES = {
    "apple",
    "apple inc.",
    "apple, inc.",
    "apple incorporated",
}

def is_aapl_sentence(span):
    """
    Decide if a sentence is about Apple stock / company.
    Heuristics:
      - contains ticker 'AAPL'
      - or has ORG/PRODUCT entity with Apple name
    """
    text_lower = span.text.lower()

    # Check explicit ticker mention
    if "aapl" in text_lower or "apple" in text_lower:
        return True

    # Check NER entities
    for ent in span.ents:
        if ent.label_ in ("ORG", "PRODUCT"):
            if ent.text.lower() in APPLE_NAMES:
                return True

    return False

@udf("string")
def split_sentences(text):
    if text is None or text == "":
        return None
    
    # Split on sentences
    doc = nlp(text)
    return '|'.join([sent.text.strip() for sent in doc.sents if is_aapl_sentence(sent)])

#### Implementation

In [0]:
news_df = news_df.withColumn('body', split(split_sentences('body'), r'\|')).withColumn('teaser', split(split_sentences('teaser'), r'\|'))

news_df.limit(10).display()

### Financial News Snapshot

In [0]:
news_df.write.format("delta").mode("overwrite").saveAsTable("aapl_news_curated")

## Financial Price Data

In [0]:
price_df = spark.read.json(f"dbfs:{VOLUME}aapl_price.json").select(col('t').alias('date'), col('o').alias('open'), col('h').alias('high'), col('l').alias('low'), col('c').alias('close'), col('v').alias('volume')) \
    .withColumn('date', to_date(to_timestamp('date', "yyyy-MM-dd'T'HH:mm:ss'Z'"))).orderBy('date')
price_df.limit(10).display()

### Feature Engineering

#### Technical Indicator Functions

In [0]:
from ta.trend import SMAIndicator
from ta.volume import AccDistIndexIndicator
import pandas as pd

@pandas_udf("double")
def sma_indicator(series : pd.Series) -> pd.Series:
    window : int = 14
    return SMAIndicator(series, window).sma_indicator()

@pandas_udf("double")
def adi_indicator(high: pd.Series,
    low: pd.Series,
    close: pd.Series,
    volume: pd.Series) -> pd.Series:
    return AccDistIndexIndicator(high=high,
        low=low,
        close=close,
        volume=volume).acc_dist_index()

@pandas_udf("double")
def percent_change(series : pd.Series) -> pd.Series:
    return series.pct_change()

#### Implementation

In [0]:
price_df = price_df.withColumn('sma', sma_indicator(col('close'))).withColumn('adi', adi_indicator(col('high'), col('low'), col('close'), col('volume'))).withColumn('ret_1d', (percent_change(col('close')) > 0).cast('int')).select('date', 'close', 'sma', 'adi', 'ret_1d')
price_df.limit(10).display()

### Financial Price Snapshot

In [0]:
price_df.write.format("delta").mode("overwrite").saveAsTable("aapl_price_curated") 

# Sentiment Analysis

## Local Model Cache

In [0]:
from huggingface_hub import snapshot_download
import os

local_tmp = "/tmp/hf_models/finbert"
os.makedirs(local_tmp, exist_ok=True)

# Download HF snapshot into a normal local folder
model_tmp = snapshot_download(
    "ProsusAI/finbert",
    local_dir=local_tmp
)

## FinBert Pipeline

In [0]:
HF_CACHE_DIR = model_tmp

from transformers import pipeline
import torch
device = 0 if torch.cuda.is_available() else -1

_classifier = None

def get_finbert():
    global _classifier
    if _classifier is None:
        _classifier = pipeline("text-classification", model=HF_CACHE_DIR,
    tokenizer=HF_CACHE_DIR, top_k=1, device=device)
    return _classifier

## Sentiment Analysis Function

In [0]:
import builtins
from collections import Counter
import numpy as np

def classify_text(text) -> dict[str, float] | None:

    if (
        text is None 
    or (isinstance(text, str) and text.strip() == "") 
    or (isinstance(text, np.ndarray) and len(text) == 1 and text[0].strip() == "")
    ):
        return None

    clf = get_finbert()  # model created on worker the first time

    if isinstance(text, str):
        text = [text]

    top_labels = []
    top_label_scores = []  # list of (label, score)

    for sent in text:
        try:
            # run FinBERT on the list of sentences
            preds = clf(
                sent
            )
        except:
            return None
    
        for per_sentence in preds:
            # per_sentence is a list like:
            # [{"label": "positive", "score": ...}, {"label": "negative", ...}, ...]
            (label, score) = next((item["label"], float(item["score"])) for item in per_sentence)
            top_labels.append(label)
            top_label_scores.append((label, score))


    all_labels = list(set(top_labels))
    all_labels_dict = [{label : np.mean([score for lbl, score in top_label_scores if lbl == label])} for label in all_labels]
    return {k : v for d in all_labels_dict for k, v in d.items()}

## Implementation

In [0]:
pd_df = news_df.toPandas()
pd_df['sentiment_body'] = pd_df['body'].apply(classify_text)
pd_df['sentiment_title'] = pd_df['title'].apply(classify_text)
pd_df['sentiment_teaser'] = pd_df['teaser'].apply(classify_text)
news_df = spark.createDataFrame(pd_df)
news_df.limit(10).display()

## Sentiment Collection

In [0]:
non_empty_text = lambda x: x.isNotNull() & (x != "")
non_empty_score = lambda x : x.score.isNotNull()
news_df = news_df \
       .withColumn('date', to_date('date')) \
       .groupBy('date').agg(
    filter(collect_list('title'), non_empty_text).alias('title'),
    filter(flatten(collect_list('teaser')), non_empty_text).alias('teaser'),
    filter(flatten(collect_list('body')), non_empty_text).alias('body'),
    filter(
       array(
       struct(
              avg('sentiment_title.positive').alias('score'), lit('positive').alias('label')
              ),
       struct(
              avg('sentiment_title.neutral').alias('score'), lit('neutral').alias('label')
              ),
       struct(
              avg('sentiment_title.negative').alias('score'), lit('negative').alias('label')
              )
       ),
       non_empty_score
       ).alias('sentiment_title'),
    filter(
           array(
           struct(
                  avg('sentiment_teaser.positive').alias('score'), lit('positive').alias('label')
                  ),
           struct(
                  avg('sentiment_teaser.neutral').alias('score'), lit('neutral').alias('label')
                  ),
           struct(
                  avg('sentiment_teaser.negative').alias('score'), lit('negative').alias('label')
                  )
           ),
           non_empty_score
           ).alias('sentiment_teaser'),
    filter(
           array(
           struct(
                  avg('sentiment_body.positive').alias('score'), lit('positive').alias('label')
                  ),
           struct(
                  avg('sentiment_body.neutral').alias('score'), lit('neutral').alias('label')
                  ),
           struct(
                  avg('sentiment_body.negative').alias('score'), lit('negative').alias('label')
                  )
           ),
           non_empty_score
           ).alias('sentiment_body')
).withColumn('sentiment_title', when(size('sentiment_title') > 0, array_max('sentiment_title')).otherwise(lit(None).cast("struct<score:double,label:string>"))) \
       .withColumn('sentiment_teaser', when(size('sentiment_teaser') > 0, array_max('sentiment_teaser')).otherwise(lit(None).cast("struct<score:double,label:string>"))) \
       .withColumn('sentiment_body', when(size('sentiment_body') > 0, array_max('sentiment_body')).otherwise(lit(None).cast("struct<score:double,label:string>"))) \
              .withColumn('sentiment_title_label', col('sentiment_title.label')) \
                     .withColumn('sentiment_title_score', col('sentiment_title.score')) \
              .withColumn('sentiment_teaser_label', col('sentiment_teaser.label')) \
                     .withColumn('sentiment_teaser_score', col('sentiment_teaser.score')) \
              .withColumn('sentiment_body_label', col('sentiment_body.label')) \
              .withColumn('sentiment_body_score', col('sentiment_body.score')) \
                     .select('date', 'title', 'teaser', 'body', 'sentiment_title_label', 'sentiment_title_score', 'sentiment_teaser_label', 'sentiment_teaser_score', 'sentiment_body_label', 'sentiment_body_score')
news_df.limit(10).display()

## Snapshot

In [0]:
news_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("aapl_news_sentiment")

# ML DataSet

## Price & News Data Join

In [0]:
df = price_df.join(news_df, on='date', how='left').select('date', 'close', 'sma', 'adi', 'sentiment_title_label', 'sentiment_title_score', 'sentiment_body_label', 'sentiment_body_score', 'ret_1d').dropna().orderBy('date')
df.limit(10).display()

## Snapshot

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("aapl_ml_data")

# Machine Learning

## Train Test Split

In [0]:
pd_df = df.orderBy('date').toPandas().set_index('date')

total = pd_df.index.size
split_idx = total - int(total * 0.2)

pd_df_wo_sent = pd_df[['close', 'sma', 'adi', 'ret_1d']]

pd_df_w_sent = pd_df[['close', 'sma', 'adi', 'sentiment_title_label', 'sentiment_title_score', 'sentiment_body_label', 'sentiment_body_score', 'ret_1d']]

X_wo_sent = pd_df_wo_sent.drop('ret_1d', axis=1)
y_wo_sent = pd_df_wo_sent['ret_1d']

X_w_sent = pd_df_w_sent.drop('ret_1d', axis=1)
y_w_sent = pd_df_w_sent['ret_1d']


X_train_wo_sent, X_test_wo_sent = X_wo_sent.iloc[:split_idx], X_wo_sent.iloc[split_idx + 1:]
y_train_wo_sent, y_test_wo_sent = y_wo_sent.iloc[:split_idx], y_wo_sent.iloc[split_idx + 1:]

X_train_w_sent, X_test_w_sent = X_w_sent.iloc[:split_idx], X_w_sent.iloc[split_idx + 1:]
y_train_w_sent, y_test_w_sent = y_w_sent.iloc[:split_idx], y_w_sent.iloc[split_idx + 1:]


## Model Initialization

In [0]:
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(
    [
        ('onehot', OneHotEncoder(sparse_output=False), ['sentiment_title_label', 'sentiment_body_label'])
    ]
)

clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="gpu_hist",        
    predictor="gpu_predictor",     
    n_estimators=300,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=0,
)


## Fit without Sentiment Analysis

In [0]:
clf.fit(X_train_wo_sent, y_train_wo_sent)

train_pred_wo_sent = clf.predict(X_train_wo_sent)

test_pred_wo_sent = clf.predict(X_test_wo_sent)

## Fit with Sentiment Analysis

In [0]:
ct.fit(X_train_w_sent)

X_train_w_sent_ct = ct.transform(X_train_w_sent)
X_test_w_sent_ct = ct.transform(X_test_w_sent)

clf.fit(X_train_w_sent_ct, y_train_w_sent)

train_pred_w_sent = clf.predict(X_train_w_sent_ct)

test_pred_w_sent = clf.predict(X_test_w_sent_ct)


## Metrics Reporting

In [0]:
from sklearn.metrics import accuracy_score

print("XGBoost without Sentiment Analysis")
print("Training Accuracy:", accuracy_score(y_train_wo_sent, train_pred_wo_sent) * 100)
print("Testing Accuracy:", accuracy_score(y_test_wo_sent, test_pred_wo_sent) * 100)
print("")
print("XGBoost with Sentiment Analysis")
print("Training Accuracy:", accuracy_score(y_train_w_sent, train_pred_w_sent) * 100)
print("Testing Accuracy:", accuracy_score(y_test_w_sent, test_pred_w_sent) * 100)

# Resources

1. [https://arxiv.org/pdf/2306.02136](https://arxiv.org/pdf/2306.02136)